# Notebook to demo convexhull by group.

In [ ]:
import math
import os

import geofunctions as S
import geopandas as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
import xyzservices.providers as xyz
from pyspark.ml.stat import Summarizer

In [ ]:
df = (
    spark.range(20)
    .withColumn("x", F.rand() * 10000)
    .withColumn("y", F.rand() * 10000)
    .withColumn("group", F.when(F.rand() < 0.5, 1).otherwise(2))
    .select("group", S.st_point("x", "y").alias("pt"))
    .groupBy("group")
    .agg(F.collect_list("pt").alias("pts"))
    .select("group", S.st_convexhull(S.st_multipoint("pts")).alias("geometry"))
)

In [ ]:
pdf = df.toPandas()
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:3857")
_ = gdf.sindex

In [ ]:
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)